# Lab - Your First Claude Request

Three short tasks. By the end you will have sent a request to Claude, read what comes
back, and seen why your code has to check `stop_reason` before a reply reaches anyone.

| Task | What you learn |
| --- | --- |
| 1 | Send one request: model, max_tokens, effort and messages |
| 2 | Read the reply text, `stop_reason` and `usage` |
| 3 | `max_tokens` is a ceiling - check `stop_reason` before you use the reply |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

It is the only cell in this lab that differs from a real project: the first two lines
load the simulator. Everything below them is ordinary Claude API code.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# The customer's question, with our return policy and the answer length we want.
QUESTION = (
    "A customer asks: can I return a jacket after 20 days? "
    "Our policy allows returns within 30 days. "
    "Answer the customer in two sentences."
)

---

## Task 1 - Make your first Claude request

In [ ]:
# ============================================================
# TASK 1 - Make your first Claude request
# ============================================================
#
# WHAT TO DO
#   Send the customer's question to Claude and print the reply.
#
# WHY IT MATTERS
#   Every feature of a Claude application starts with this one
#   call. Tools, agents and the agentic loop later in this
#   course are all this same call, repeated.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The request itself is one method call: client dot messages
#   dot create".
#
# HOW TO DO IT
#   The call takes four things. Replace each ... with the value
#   named beside it:
#
#       model          the MODEL constant from the setup cell
#       max_tokens     300 - the most tokens Claude may write
#       effort         "low" - a simple question does not need
#                      Claude to think before it answers
#       content        the QUESTION constant
#
#   Assign the WHOLE result to `response`. The next task reads
#   several fields from it.
# ============================================================

# TODO: replace each ... below with the value named beside it
response = client.messages.create(
    model=...,                        # the MODEL constant
    max_tokens=...,                   # 300
    output_config={"effort": ...},    # "low", in quotes
    messages=[
        {"role": "user", "content": ...},   # the QUESTION constant
    ],
)

print(response.content[0].text)

check("first_request", response=response)

---

## Task 2 - Read the text, `stop_reason` and `usage`

In [ ]:
# ============================================================
# TASK 2 - Read the text, stop_reason and usage
# ============================================================
#
# WHAT TO DO
#   Pull four values out of the response from Task 1.
#
# WHY IT MATTERS
#   The response is a Message object with several fields.
#   content is a list of blocks; for a plain answer it holds one
#   text block, and its text is what the customer will see.
#   stop_reason says why Claude stopped writing. usage counts
#   the tokens you sent and the tokens Claude wrote - which is
#   what you pay for.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now let's read the response".
#
# HOW TO DO IT
#   The first line is done for you as an example: it reaches
#   into the object with dots. Finish the other three the same
#   way. Do not type the numbers you see printed - read them
#   from the object, the way your backend would.
#
#       stop_reason     response.stop_reason
#       input_tokens    response.usage.input_tokens
#       output_tokens   response.usage.output_tokens
# ============================================================

# TODO: replace each ... below with the field named beside it
reply_text = response.content[0].text
stop_reason = ...     # response.stop_reason
input_tokens = ...    # response.usage.input_tokens
output_tokens = ...   # response.usage.output_tokens

print(reply_text)
print("stop_reason:", stop_reason)
print("usage:", input_tokens, "in,", output_tokens, "out")

check("response_object", response=response, reply_text=reply_text,
      stop_reason=stop_reason, input_tokens=input_tokens, output_tokens=output_tokens)

---

## Task 3 - `max_tokens` is a ceiling: check `stop_reason`

In [ ]:
# ============================================================
# TASK 3 - max_tokens is a ceiling: check stop_reason
# ============================================================
#
# WHAT TO DO
#   Send the same question with max_tokens=20, then finish the
#   small function that decides whether a reply may be shown.
#
# WHY IT MATTERS
#   It is tempting to think of max_tokens as the length you
#   want. It is not. It is a ceiling, and Claude stops the
#   moment it reaches it - even in the middle of a sentence.
#   Nothing fails: stop_reason simply says "max_tokens" instead
#   of "end_turn". If ShopAssist showed that text, the customer
#   would see half a sentence.
#
#   So the practical rule: check stop_reason before you use the
#   reply. "end_turn" means the answer is complete. Anything
#   else, and the application raises the limit or handles the
#   cut instead of showing the fragment.
#
# WHERE TO SEE IT IN THE LECTURE
#   "It is tempting to think of max tokens as the length we
#   want".
#
# HOW TO DO IT
#   Two blanks:
#
#       max_tokens    20 - far too few for two sentences
#       "end_turn"    the stop_reason of a complete answer, in
#                     quotes
# ============================================================

# TODO: replace each ... below with the value named beside it
short = client.messages.create(
    model=MODEL,
    max_tokens=...,   # 20 - far too few for two sentences
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": QUESTION}],
)


def safe_reply(resp):
    if resp.stop_reason == ...:   # "end_turn" - the reply is complete
        return resp.content[0].text
    return None   # cut off - never show a fragment to a customer

print("stop_reason:", short.stop_reason)
print("what came back:", repr(short.content[0].text))
print()
print("shown to the customer (20 tokens): ", safe_reply(short))
print("shown to the customer (300 tokens):", safe_reply(response))

check("stop_reason", short=short, long=response, safe_reply=safe_reply)

---

## Done

One request, read three ways: the text for the customer, `stop_reason` for your code,
and `usage` for the bill. And the rule that comes with it: `max_tokens` is a ceiling,
so check `stop_reason` before any reply reaches a customer.

Two commands worth knowing if you want to poke around:

| Call | What it does |
| --- | --- |
| `lab_info()` | what the simulator does and does not reproduce |
| `explain(response)` | why the simulator returned that particular reply |